In [1]:
from openai import AzureOpenAI
import tiktoken
import json
import pandas as pd
import os

In [2]:
with open('C:/Users/Dulin/Dropbox/UTHealth/SAFElab/LLM/key.txt', "r", encoding="UTF-8") as file:
    key = file.read()
TEMPERATURE=0
MAX_TOKENS=6000
# azure_openai_key = ''
# azure_openai_endpoint = 'https://cursorplayground-002.openai.azure.com/'
# azure_api_verison = '2024-02-15-preview'
# azure_openai_model = 'cursor_35_16k' 
azure_openai_key = key
azure_openai_endpoint = 'https://ontology-kz.openai.azure.com/'
azure_api_verison = '2024-02-15-preview'
azure_openai_model = 'mCodegpt_gpt_4_32k' 

azure_endpoint = "https://cursorplayground-002.openai.azure.com/openai/deployments/cursor_35_16k/chat/completions?api-version=2024-02-15-preview"
azure_tpm= 120000
azure_rate_limit = 720
input_cost_per_million = 0.50
output_cost_per_million = 1.50

client = AzureOpenAI(api_key=azure_openai_key, azure_endpoint=azure_openai_endpoint, api_version=azure_api_verison)

In [3]:

### Split the notes to inclusion and exclusion criteria seperately

input_dir = './input'
df = pd.DataFrame(columns=['NOTES'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            df.loc[file_name] = [content]
abstract_list = df['NOTES'].tolist()
print("Abstracts have been successfully read into a list.")

### 

def process_abstracts_and_get_tool_arguments(abstract_list, client, tools):
    global TEMPERATURE
    global MAX_TOKENS
    tool_arguments_df = pd.DataFrame(columns=['Abstract Index', 'Arguments', 'Cost', 'Error'])
    processed_abstracts = []  

    for index, abstract in enumerate(abstract_list, start=1):
        print("\n" + "="*50)
        print(f"Processing abstract {index}...")
        print("="*50 + "\n")
        abstract_str = str(abstract) if not isinstance(abstract, str) else abstract
        clean_text = abstract_str.replace("  ", " ").replace("\n", "; ").replace(';',' ')

        ### Imporve the prompt with details
        prompt_template = "Extract information from the following clinical note: <content>"
        prompt = prompt_template.replace("<content>", clean_text)
        ### 
        messages = [
          {"role": "system", "content": "You will help extract information from documents."},
          {"role": "user", "content": prompt}
          ]
        try:
            completion = client.chat.completions.create(
                model= azure_openai_model,
                messages=messages,
                tools=tools,
                tool_choice="auto",
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS
              )
            tool_calls = completion.choices[0].message.tool_calls
            arguments_list = []
        
            for tool_call in tool_calls:
                arguments_str = tool_call.function.arguments
                arguments = json.loads(arguments_str)
                arguments_list.append(arguments)
            prompt_tokens = completion.usage.prompt_tokens
            completion_tokens = completion.usage.completion_tokens
            total_tokens = completion.usage.total_tokens

            input_cost_per_token = input_cost_per_million / 1_000_000
            output_cost_per_token = output_cost_per_million / 1_000_000
            total_input_cost = prompt_tokens * input_cost_per_token
            total_output_cost = completion_tokens * output_cost_per_token
            total_cost = total_input_cost + total_output_cost
            print(f"Cost for processing abstract {index}: {total_tokens} tokens, Total Cost: ${total_cost:.5f}")
            new_row = pd.DataFrame([{'Abstract Index': index, 'Arguments': arguments_list, 'Cost': total_cost, 'Error': None}])
            tool_arguments_df = pd.concat([tool_arguments_df, new_row], ignore_index=True)

            processed_abstracts.append({'index': index, 'abstract': abstract_str, 'arguments': arguments_list, 'cost': total_cost})

            with open('processed_abstracts.json', 'w') as f:
                json.dump(processed_abstracts, f, indent=4)

        except Exception as e:
            print(f"An error occurred while processing abstract {index}: {e}")
            new_row = pd.DataFrame([{'Abstract Index': index, 'Arguments': None, 'Cost': None, 'Error': str(e)}])
            tool_arguments_df = pd.concat([tool_arguments_df, new_row], ignore_index=True)
            processed_abstracts.append({'index': index, 'abstract': abstract_str, 'error': str(e)})
            with open('processed_abstracts.json', 'w') as f:
                json.dump(processed_abstracts, f, indent=4)

    return tool_arguments_df


def generate_argument_entries(arguments):
    entries = []
    try:
        for arg in arguments:
            for key, value in arg.items():
                if isinstance(value, str) and value:
                    entry = f'"{key}": "{value}"'
                elif isinstance(value, (int, float)) and value != 0:
                    entry = f'"{key}": {value}'
                elif isinstance(value, str):
                    entry = f'"{key}": ""'
                else:
                    entry = f'"{key}": {value}'
                entries.append(entry)
    except:
        print("TypeError: 'NoneType' object is not iterable")
        return None
    return "\n".join(entries)

Abstracts have been successfully read into a list.


In [ ]:
with open("GEARBOx.json", "r") as file:
    tool = json.load(file)
    
tools = [tool]
tool_arguments_df = process_abstracts_and_get_tool_arguments(abstract_list, client, tools)

from datetime import datetime
output_excel_path = f'output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'

output_df = tool_arguments_df.apply(lambda x: json.dumps(x.to_dict()), axis=1)

output_df = tool_arguments_df.apply(lambda x: generate_argument_entries(x['Arguments']), axis=1)

df['Result'] = output_df.tolist()

df.to_excel(output_excel_path)
print(f"Data has been written to {output_excel_path}")

In [ ]:
output_df = tool_arguments_df.apply(lambda x: json.dumps(x.to_dict()), axis=1)

output_df = tool_arguments_df.apply(lambda x: generate_argument_entries(x['Arguments']), axis=1)

df['Result'] = output_df.tolist()

df.to_excel(output_excel_path)
print(f"Data has been written to {output_excel_path}")
